# Tahap 2 sampai 4: Eksplorasi, Praproses, Validasi, dan Baseline
**Proyek:** Peramalan PM2.5 Beijing (Muhammad Alif Qadri, D082261018)

Tujuan bagian ini sederhana: **mengenal data, menyiapkannya, dan menetapkan acuan (*baseline*)** sebelum model lanjutan dibuat.

**Istilah yang dipakai** (supaya tidak tertukar):

| Istilah | Arti |
|---|---|
| Stasiun latih | 9 stasiun yang datanya dipakai untuk melatih model |
| Stasiun asing | 3 stasiun yang **belum pernah dilihat model** saat dilatih (Dingling, Wanliu, Shunyi) |
| Periode latih | 80% waktu awal (sampai 2016-05-12 18:00), sama untuk semua stasiun |
| Periode uji | 20% waktu akhir (mulai 2016-05-12 19:00), sama untuk semua stasiun |
| Data latih | stasiun latih pada periode latih |
| Data uji | 12 stasiun pada periode uji, dilaporkan terpisah untuk stasiun latih dan stasiun asing |
| Tidak dipakai | stasiun asing pada periode latih (hanya dilihat untuk memilih stasiun asing dan menggambarkan pergeseran) |

Aturan yang dipegang sepanjang *notebook*:
1. Data uji disisihkan **di awal** dan tidak dilihat.
2. Model meramal PM2.5 **24 jam ke depan**, jadi fitur hanya boleh memakai data sampai jam sekarang.
3. Tidak ada langkah yang memakai nilai dari masa depan (misalnya interpolasi).
4. Validasi berurutan waktu memakai jarak aman 24 jam antara data latih dan data validasi. Lipatan dibuat dari **jam unik**, bukan dari baris gabungan stasiun.

Nomor bagian mengikuti struktur *notebook* dari dosen:

| No | Isi |
|---|---|
| 00 | Pengaturan |
| 01 | Rumusan masalah (ringkas) |
| 02 | Muat data, sisihkan data uji dan stasiun asing, lalu eksplorasi (EDA) |
| 03 | Pembagian data dan skema validasi |
| 04 | Fitur, *pipeline* praproses, *baseline*, dan pemeriksaan kesesuaian desain |

Status setiap keputusan:
* **[Panduan]** dari dosen: data berurutan waktu tidak diacak, `TimeSeriesSplit`, *naive forecast* sebagai *baseline*, data uji dibuka satu kali.
* **[Keputusan LIP]** bukan aturan dosen: horizon 24 jam, nilai ekstrem dipertahankan (14 Nov 2026), tanpa *forward fill* (15 Nov 2026, dasarnya dihitung ulang di 04.6), 9 stasiun latih dan 3 stasiun asing dengan satu batas waktu untuk semua stasiun, tiga stasiun asing = Dingling, Wanliu, Shunyi (dikunci, dasarnya aturan di 02.2), dan identitas stasiun tidak dipakai sebagai fitur (16 Nov 2026).
* **Belum dikonfirmasi dosen:** desain 9 stasiun latih dan 3 stasiun asing.

Catatan urutan: data uji disisihkan **di awal bagian 02**, sebelum eksplorasi, sesuai Kode 1 panduan
("data uji disisihkan sekarang, sebelum eksplorasi lebih jauh"). Bagian 03 merinci aturan pembagiannya.

## 00. Pengaturan

In [ ]:
# Sumber data:
# 1) Google Drive pemilik (folder "Dataset PM25 Beijing"), jika ada.
# 2) Folder Data/Data Mentah di repo ini (dijalankan dari akar repo atau dari folder Notebook).
# 3) Jika keduanya tidak ada (misalnya dijalankan dosen), repo GitHub publik diunduh.
import os, subprocess
REPO = "https://github.com/LippyyDev/Tugas-Machine-Learning-PM25-Beijing"

FOLDER = os.environ.get("FOLDER_DATA")
if FOLDER is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        calon = "/content/drive/MyDrive/Dataset PM25 Beijing"
        FOLDER = calon if os.path.isdir(calon) else None
    except Exception:
        pass
if FOLDER is None:
    for calon in ("Data/Data Mentah", "../Data/Data Mentah"):
        if os.path.isdir(calon):
            FOLDER = calon                                # dijalankan dari dalam repo
            break
    else:
        if not os.path.isdir("repo_pm25"):
            subprocess.run(["git", "clone", "--depth", "1", REPO, "repo_pm25"], check=True)
        FOLDER = "repo_pm25/Data/Data Mentah"

# Keluaran meniru struktur repo: Hasil/Gambar, Hasil/Tabel, Data/Data Olahan.
# Di repo, akar = folder yang memuat "Data". Di Drive, akar = folder data itu sendiri.
if os.path.basename(os.path.normpath(FOLDER)) == "Data Mentah":
    AKAR = os.path.abspath(os.path.join(FOLDER, "..", ".."))
else:
    AKAR = os.path.abspath(FOLDER)
GAMBAR = os.path.join(AKAR, "Hasil", "Gambar")
TABEL = os.path.join(AKAR, "Hasil", "Tabel")
OLAHAN = os.path.join(AKAR, "Data", "Data Olahan")
for d in (GAMBAR, TABEL, OLAHAN):
    os.makedirs(d, exist_ok=True)
print("Folder data   :", FOLDER)
print("Keluaran disimpan di Hasil/Gambar, Hasil/Tabel, dan Data/Data Olahan di bawah folder:",
      os.path.basename(AKAR))

In [ ]:
import glob, sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

SEED = 42
np.random.seed(SEED)
HORIZON = 24                     # jam ke depan yang diramal
KOLOM_ANGKA = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

# Warna grafik: biru = data utama, oranye = sorotan, abu-abu = pembanding
BIRU, ORANYE, ABU = "#2a78d6", "#eb6834", "#b4b2a9"
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10, "figure.dpi": 100})

# Nama kolom yang mudah dibaca untuk grafik
NAMA_KOLOM = {"PM2.5": "PM2.5 (partikel halus)", "PM10": "PM10 (partikel kasar)", "SO2": "SO2", "NO2": "NO2", "CO": "CO",
              "O3": "O3", "TEMP": "TEMP (suhu)", "PRES": "PRES (tekanan udara)", "DEWP": "DEWP (titik embun)",
              "RAIN": "RAIN (hujan)", "WSPM": "WSPM (kecepatan angin)", "wd": "wd (arah angin)"}

def judul(ax, utama, keterangan=None):
    # judul tebal di kiri, keterangan cara membaca di bawahnya
    ax.set_title(utama, loc="left", fontsize=12, fontweight="bold", pad=28 if keterangan else 10)
    if keterangan:
        ax.text(0, 1.025, keterangan, transform=ax.transAxes, fontsize=9, color="#52514e", va="bottom")

def simpan_gambar(nama):
    plt.tight_layout()
    plt.savefig(os.path.join(GAMBAR, nama), dpi=150)
    plt.show()

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__,
      "| matplotlib", matplotlib.__version__)

## 01. Rumusan masalah (ringkas)
* **Target:** konsentrasi PM2.5 pada **24 jam ke depan** di setiap stasiun. Satu model dilatih dari gabungan **9 stasiun latih**.
* **Fitur:** hanya data yang sudah tersedia saat ramalan dibuat (polutan, cuaca, waktu), dihitung **per stasiun**, tanpa mencampur deret antar stasiun.
* **Pengujian:** pada 20% periode terakhir, dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun asing (stasiun yang tidak pernah dilihat model).
* **Metrik utama:** RMSE, karena galat besar saat polusi tinggi paling merugikan. MAE sebagai pendamping.

Rumusan lengkap ada di proposal.

## 02. Muat data dan eksplorasi
### 02.1 Muat data
Ada 12 berkas CSV, satu per stasiun. Semuanya digabung menjadi satu tabel.

In [ ]:
berkas = sorted(glob.glob(os.path.join(FOLDER, "*.csv")))
df = pd.concat([pd.read_csv(f) for f in berkas], ignore_index=True)
df["waktu"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.sort_values(["station", "waktu"]).reset_index(drop=True)

per_stasiun = df.groupby("station")["waktu"].agg(baris="size", jam_unik="nunique")
assert (per_stasiun["baris"] == per_stasiun["jam_unik"]).all()      # tidak ada jam ganda dalam satu stasiun
print(f"{len(berkas)} berkas, {df.shape[0]:,} baris, {df.shape[1]} kolom")
print(f"Baris per stasiun: {per_stasiun['baris'].min():,} sampai {per_stasiun['baris'].max():,}")
print("Rentang waktu:", df["waktu"].min(), "sampai", df["waktu"].max())
df.head(3)

### 02.2 Sisihkan data uji dan stasiun asing (sebelum eksplorasi)
Dua pemotongan dibuat di sini, sebelum eksplorasi:

1. **Menurut waktu:** data uji = **20% periode terakhir**, tanpa diacak. Batas waktunya **sama untuk semua stasiun** (satu stempel waktu), supaya setiap stasiun diuji pada periode yang sama dan tidak ada jam uji yang ikut terlatih lewat stasiun lain. Stasiun Beijing saling berkorelasi tinggi pada jam yang sama, jadi memotong per stasiun akan membocorkan masa depan.
2. **Menurut stasiun:** 3 dari 12 stasiun **asing**. Model tidak pernah dilatih atau divalidasi dengan baris dari stasiun itu. Hanya 20% periode terakhirnya yang dipakai, sebagai uji "stasiun yang belum pernah dilihat" (pergeseran distribusi, tantangan wajib ketiga).

| Blok | Stasiun | Periode | Dipakai untuk |
|---|---|---|---|
| Latih | 9 stasiun latih | 80% awal | melatih dan validasi berurutan waktu |
| Uji stasiun latih | 9 stasiun latih | 20% akhir | uji akhir, stasiun yang dikenal model |
| Tidak dipakai | 3 stasiun asing | 80% awal | tidak melatih dan tidak memvalidasi |
| Uji stasiun asing | 3 stasiun asing | 20% akhir | uji akhir, stasiun yang belum pernah dilihat model |

**Tiga stasiun asing: Dingling, Wanliu, Shunyi [Keputusan LIP, 16 Nov 2026, dikunci].** Dasarnya aturan berikut:
* Stasiun diurutkan menurut jarak distribusi PM2.5 (jarak Wasserstein) terhadap gabungan 11 stasiun lain, hanya dari periode **sebelum** batas waktu.
* Yang dijadikan stasiun asing: urutan ke-1 (paling berbeda), ke-6 (sedang), dan ke-12 (paling mirip), supaya uji mencakup beberapa tingkat pergeseran.
* Tidak ada model yang dilatih untuk memilih, dan data uji tidak dilihat.

**Batasnya [jujur]:** pemilihan memakai data periode latih dari stasiun yang kemudian dijadikan stasiun asing, jadi hasil uji pada tiga stasiun itu adalah tiga contoh, bukan perkiraan umum untuk "stasiun mana pun".

Sel di bawah menghitung ulang aturan itu dan memastikan hasilnya sama dengan daftar yang dikunci. Mengganti daftar berarti mengubah keputusan, dan seluruh *notebook* harus dijalankan ulang dari awal.

In [ ]:
jam_unik = np.sort(df["waktu"].unique())
assert (np.diff(jam_unik) == np.timedelta64(1, "h")).all()    # jam berurutan tanpa lompatan, jadi 24 baris unik = 24 jam
BATAS = pd.Timestamp(jam_unik[int(len(jam_unik) * 0.8)])      # jam pertama periode uji, sama untuk semua stasiun

pra = df[df["waktu"] < BATAS]      # semua 12 stasiun sebelum BATAS: HANYA untuk memilih stasiun asing dan menggambarkan pergeseran
print("Batas data uji (jam pertama periode uji):", BATAS)

In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance

# Langkah 1: jarak distribusi PM2.5 tiap stasiun terhadap 11 stasiun lain (hanya periode sebelum BATAS)
tab_dist = []
for s, g in pra.groupby("station"):
    v = g["PM2.5"].dropna().values
    lain = pra.loc[pra["station"] != s, "PM2.5"].dropna().values
    tab_dist.append(dict(stasiun=s, rata_rata=v.mean(), median=np.median(v), sd=v.std(), kuantil90=np.quantile(v, .9),
                         hilang_pct=100 * g["PM2.5"].isna().mean(),
                         wasserstein=wasserstein_distance(v, lain), KS=ks_2samp(v, lain).statistic))
tab_dist = pd.DataFrame(tab_dist).set_index("stasiun")
piv = pra.pivot_table(index="waktu", columns="station", values="PM2.5")
tab_dist["korelasi_dg_lainnya"] = [piv[s].corr(piv.drop(columns=s).mean(axis=1)) for s in tab_dist.index]
tab_dist = tab_dist.sort_values("wasserstein", ascending=False)
tab_dist.round(3)

In [ ]:
# Langkah 2: daftar stasiun asing yang DIKUNCI, lalu dicek terhadap aturan
STASIUN_ASING = ["Dingling", "Wanliu", "Shunyi"]    # DIKUNCI: keputusan LIP 16 Nov 2026
URUTAN_ASING = (1, 6, 12)                           # dasar aturan: urutan jarak Wasserstein dari terbesar (1) ke terkecil (12)

dari_aturan = [tab_dist.index[i - 1] for i in URUTAN_ASING]
assert sorted(dari_aturan) == sorted(STASIUN_ASING), (
    f"Aturan urutan {URUTAN_ASING} menghasilkan {dari_aturan}, berbeda dari daftar yang dikunci {STASIUN_ASING}")
STASIUN_LATIH = [s for s in sorted(df["station"].unique()) if s not in STASIUN_ASING]
assert len(STASIUN_ASING) == 3 and set(STASIUN_ASING) <= set(tab_dist.index)
assert len(STASIUN_LATIH) == 9 and not set(STASIUN_LATIH) & set(STASIUN_ASING)

tab_dist["kelompok"] = np.where(tab_dist.index.isin(STASIUN_ASING), "asing", "latih")
tab_dist = tab_dist.round(3)
tab_dist.to_csv(os.path.join(TABEL, "t2_tabel_distribusi_stasiun.csv"))
kor_pasangan = np.median(piv.corr().values[np.triu_indices(piv.shape[1], k=1)])
print("Stasiun asing (dikunci, sama dengan hasil aturan):", STASIUN_ASING)
print("Stasiun latih  :", STASIUN_LATIH)
print(f"Median korelasi PM2.5 antar pasangan stasiun pada jam yang sama (sebelum BATAS): {kor_pasangan:.2f}")
tab_dist

In [ ]:
latih = df[df["station"].isin(STASIUN_LATIH) & (df["waktu"] < BATAS)].copy()
blok = {
    "Latih (9 stasiun latih, 80% awal)":           len(latih),
    "Uji stasiun latih (9 stasiun, 20% akhir)":    (df["station"].isin(STASIUN_LATIH) & (df["waktu"] >= BATAS)).sum(),
    "Tidak dipakai (3 stasiun asing, 80% awal)":   (df["station"].isin(STASIUN_ASING) & (df["waktu"] < BATAS)).sum(),
    "Uji stasiun asing (3 stasiun, 20% akhir)":    (df["station"].isin(STASIUN_ASING) & (df["waktu"] >= BATAS)).sum(),
}
assert sum(blok.values()) == len(df)
for nama, n in blok.items():
    print(f"{nama:<46}{n:>9,} baris  {n / len(df):>5.0%}")
print("\nMulai di sini, eksplorasi hanya memakai `latih` (9 stasiun latih, sebelum BATAS).")

### 02.3 Statistik deskriptif (9 stasiun latih, digabung)

In [ ]:
latih[["PM2.5", "PM10", "TEMP", "DEWP", "PRES", "WSPM", "RAIN"]].describe().round(1)

### 02.4 Sebaran PM2.5
Seberapa sering PM2.5 bernilai rendah, dan seberapa sering sangat tinggi? Ini dasar keputusan soal nilai ekstrem.

In [ ]:
pm = latih["PM2.5"].dropna()
BATAS_TAMPIL = 500                       # nilai di atas ini tidak digambar supaya bentuk utamanya terlihat
fig, ax = plt.subplots()
ax.hist(pm[pm <= BATAS_TAMPIL], bins=50, color=BIRU, edgecolor="white", linewidth=0.5)
ax.axvline(pm.median(), color=ORANYE, lw=2, label=f"Median = {pm.median():.0f} (separuh jam di bawah nilai ini)")
ax.axvline(pm.mean(), color="#0b0b0b", lw=1.5, ls="--", label=f"Rata-rata = {pm.mean():.0f}")
ax.legend(loc="upper right", frameon=False)
ax.text(0.98, 0.62, f"{(pm > BATAS_TAMPIL).sum()} jam bernilai di atas {BATAS_TAMPIL}\nTidak digambar, tetapi tetap dipakai", transform=ax.transAxes,
        ha="right", fontsize=9, color="#52514e")
judul(ax, "Sebaran PM2.5 per jam (data latih)",
      "Rata-rata lebih tinggi dari median karena ada jam yang bernilai sangat tinggi.")
ax.set(xlabel="Konsentrasi PM2.5 (µg/m³)", ylabel="Jumlah jam (9 stasiun digabung)")
simpan_gambar("t2_01_sebaran_pm25.png")
print(f"Rata-rata {pm.mean():.1f} | median {pm.median():.1f} | kemiringan (skewness) {pm.skew():.2f}")

### 02.5 Nilai ekstrem
Nilai sangat tinggi bisa berarti episode polusi nyata atau galat sensor.
Kita cek di 9 stasiun latih (data latih), lalu lihat apakah nilai itu muncul sendirian atau bersamaan di banyak stasiun.

In [ ]:
pm_semua = latih["PM2.5"]
print("Jumlah jam-stasiun dengan PM2.5:")
for ambang in [500, 700, 900]:
    print(f"  di atas {ambang}: {(pm_semua > ambang).sum():>4}")
print("  tepat 999   :", (pm_semua == 999).sum())

# Apakah nilai > 700 terjadi serempak? Hitung berapa stasiun yang juga > 300 pada jam itu.
jam_ekstrem = latih.loc[latih["PM2.5"] > 700, "waktu"].unique()
serempak = (latih[latih["waktu"].isin(jam_ekstrem)]
            .groupby("waktu")["PM2.5"].apply(lambda s: (s > 300).sum()))
print(f"\nPada {len(jam_ekstrem)} jam dengan PM2.5 > 700 di suatu stasiun, "
      f"median jumlah stasiun yang juga > 300: {serempak.median():.0f} dari {latih['station'].nunique()}")

### 02.6 Pola harian dan musiman
Kapan polusi paling tinggi? Pola ini menjadi dasar fitur kalender (jam dan bulan).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
per_jam = latih.groupby(latih["waktu"].dt.hour)["PM2.5"].mean()
a1.bar(per_jam.index, per_jam, color=BIRU, width=0.8)
judul(a1, "Rata-rata PM2.5 menurut jam dalam sehari")
a1.set(xlabel="Jam (0 = tengah malam, 23 = pukul 23.00)", ylabel="Rata-rata PM2.5 (µg/m³)")

nama_bulan = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"]
per_bulan = latih.groupby(latih["waktu"].dt.month)["PM2.5"].mean()
a2.bar([nama_bulan[m-1] for m in per_bulan.index], per_bulan, color=BIRU, width=0.8)
judul(a2, "Rata-rata PM2.5 menurut bulan")
a2.set(xlabel="Bulan", ylabel="Rata-rata PM2.5 (µg/m³)")
a2.tick_params(axis="x", rotation=45)
simpan_gambar("t2_03_pola_jam_bulan.png")

print(f"Jam tertinggi: {per_jam.idxmax()}:00 ({per_jam.max():.0f}) | "
      f"terendah: {per_jam.idxmin()}:00 ({per_jam.min():.0f})")
print(f"Bulan tertinggi: {nama_bulan[per_bulan.idxmax()-1]} ({per_bulan.max():.0f}) | "
      f"terendah: {nama_bulan[per_bulan.idxmin()-1]} ({per_bulan.min():.0f})")

### 02.7 Data bolong (nilai hilang)
Dua hal yang dicek: **berapa banyak** yang hilang, dan **berapa lama** celahnya (dihitung per stasiun, lalu digabung).
Panjang celah menentukan seberapa berbahaya tebakan pengisiannya (dibahas di 04.1).

In [ ]:
hilang = (latih.drop(columns=["No", "year", "month", "day", "hour", "station", "waktu"])
            .isna().mean().mul(100).sort_values())
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh([NAMA_KOLOM.get(k, k) for k in hilang.index], hilang,
        color=[ORANYE if k == "PM2.5" else BIRU for k in hilang.index])
for y, v in enumerate(hilang):
    ax.text(v + 0.05, y, f"{v:.1f}%", va="center", fontsize=9)
judul(ax, "Persentase jam yang datanya kosong, per kolom (data latih)",
      "Makin panjang batang, makin sering sensor tidak mencatat. Oranye = PM2.5, yang ingin diramal.")
ax.set(xlabel="Persen jam yang kosong (%)", xlim=(0, hilang.max() * 1.15))
ax.grid(axis="y", visible=False)
simpan_gambar("t2_04_nilai_hilang_per_kolom.png")
hilang.round(2).to_frame("% hilang").to_csv(os.path.join(TABEL, "t2_tabel_nilai_hilang.csv"))

In [ ]:
# Panjang setiap celah beruntun pada PM2.5, dihitung di dalam tiap stasiun
def panjang_celah(seri):
    kosong = seri.isna()
    kelompok = (kosong != kosong.shift()).cumsum()
    return kosong.groupby(kelompok).sum()[lambda s: s > 0]

celah = pd.concat([panjang_celah(g.sort_values("waktu")["PM2.5"]) for _, g in latih.groupby("station")],
                  ignore_index=True)
kelas = pd.cut(celah, [0, 1, 3, 6, 24, 10_000],
               labels=["1 jam", "2-3 jam", "4-6 jam", "7-24 jam", "lebih dari 24 jam"])
hitung = kelas.value_counts().sort_index()

fig, ax = plt.subplots()
ax.bar(hitung.index.astype(str), hitung, color=BIRU, width=0.6)
for x, y in zip(range(len(hitung)), hitung):
    ax.text(x, y, str(y), ha="center", va="bottom")
judul(ax, "Berapa lama data PM2.5 kosong sekali jalan? (data latih)",
      f"Tiap batang = jumlah kejadian kosong. {(celah <= 3).mean():.0%} kejadian hanya berlangsung 1 sampai 3 jam.")
ax.set(xlabel="Lama kosong berturut-turut", ylabel="Jumlah kejadian (9 stasiun digabung)", ylim=(0, hitung.max() * 1.12))
simpan_gambar("t2_05_panjang_celah.png")

print(f"Total celah: {len(celah)} | median {celah.median():.0f} jam | terpanjang {celah.max()} jam")
for k in [1, 3, 6]:
    print(f"  Celah <= {k} jam: {(celah <= k).mean():.0%} dari celah, "
          f"{celah[celah <= k].sum() / celah.sum():.0%} dari jam hilang")

### 02.8 Apa yang berkaitan dengan PM2.5 besok?
Grafik ini menghitung korelasi antara nilai **sekarang** (jam *t*) dengan PM2.5 **24 jam kemudian** di stasiun yang sama (digeser per stasiun, lalu digabung).
Inilah informasi yang benar-benar tersedia saat ramalan dibuat, dan dasar memilih fitur "kondisi sekarang".

In [ ]:
target_besok = latih.groupby("station")["PM2.5"].shift(-HORIZON)        # PM2.5 pada jam t+24, di stasiun yang sama
korelasi = latih[KOLOM_ANGKA].corrwith(target_besok).sort_values()

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh([NAMA_KOLOM[k] for k in korelasi.index], korelasi, color=[BIRU if v >= 0 else ORANYE for v in korelasi])
for y, v in enumerate(korelasi):
    ax.text(v + (0.008 if v >= 0 else -0.008), y, f"{round(v, 2) + 0.0:.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=9)
ax.axvline(0, color="black", lw=0.8)
judul(ax, "Nilai mana yang berhubungan dengan PM2.5 24 jam ke depan?",
      "Biru = naik bersama PM2.5 besok, oranye = berlawanan. Makin jauh dari 0, makin kuat hubungannya.")
ax.set(xlabel="Korelasi antara nilai jam ini dan PM2.5 24 jam kemudian", xlim=(korelasi.min() - 0.07, korelasi.max() + 0.07))
ax.grid(axis="y", visible=False)
simpan_gambar("t2_06_korelasi_dengan_besok.png")
korelasi.round(3)

### 02.9 Seberapa jauh masa lalu masih berguna?
Korelasi PM2.5 dengan PM2.5 beberapa jam sebelumnya. Ini membantu memilih *lag* yang dipakai sebagai fitur.

In [ ]:
jarak = [24, 25, 26, 27, 30, 36, 48, 72, 96, 120, 144, 168]   # selisih jam antara dua nilai PM2.5 yang dibandingkan
kor_lag = [latih["PM2.5"].corr(latih.groupby("station")["PM2.5"].shift(j)) for j in jarak]

fig, ax = plt.subplots()
posisi = list(range(len(jarak)))                  # jarak antar titik sama, supaya angka tidak bertumpuk
ax.plot(posisi, kor_lag, color=BIRU, lw=2, marker="o", ms=7)
for x, y in zip(posisi, kor_lag):
    ax.text(x, y + 0.012, f"{y:.2f}", ha="center", fontsize=9)
judul(ax, "Hubungan PM2.5 dengan PM2.5 beberapa jam sebelumnya (data latih)",
      "Makin kecil korelasi, makin sedikit yang dibawa nilai lama untuk meramal.")
ax.set(xlabel="Selisih waktu antara dua nilai PM2.5 yang dibandingkan (jam, skala tidak linear)", ylabel="Korelasi",
       xticks=posisi, xticklabels=jarak, ylim=(0, max(kor_lag) + 0.06))
simpan_gambar("t2_07_autokorelasi.png")
pd.Series(kor_lag, index=jarak, name="korelasi").round(3).to_frame().T

### 02.10 Perbandingan antarstasiun
Dasar untuk tantangan wajib ketiga: model diuji di stasiun yang belum pernah dilihat. Seberapa beda stasiunnya?
Tabel di bawah memakai data periode latih dari **semua 12 stasiun**, hanya untuk menggambarkan pergeseran distribusi. Model tidak melihat baris stasiun asing.

In [ ]:
ringkas = (tab_dist[["rata_rata", "median", "hilang_pct", "kelompok"]]
           .rename(columns={"hilang_pct": "persen_hilang"}).sort_values("rata_rata"))
ringkas.round(1).to_csv(os.path.join(TABEL, "t2_tabel_ringkas_stasiun.csv"))
ringkas.round(1)

## 03. Pembagian data dan skema validasi
Aturan ujian model, sesuai panduan untuk data berurutan waktu:

1. **Data uji** = 20% periode terakhir (disisihkan di 02.2), untuk 12 stasiun dengan satu batas waktu. Disimpan, dan dibuka **satu kali** di akhir proyek. Dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun asing.
2. **Data latih** = 80% periode awal dari 9 stasiun latih. Semua pemilihan model dan *tuning* memakai `TimeSeriesSplit(n_splits=5, gap=24)` di sini.
3. **Stasiun asing** tidak dipakai untuk melatih, memvalidasi, maupun menyetel apa pun.
4. Data **tidak diacak**. Setiap *fold* melatih dengan masa lalu dan memvalidasi dengan periode sesudahnya.
5. **Jarak aman 24 jam** (`gap=24`). Target sebuah baris adalah PM2.5 24 jam sesudahnya. Tanpa jarak, target baris latih terakhir jatuh di periode validasi, padahal nilai itu juga muncul sebagai fitur *lag* di data validasi. Jarak 24 jam menutup celah ini.
6. **Lipatan dibuat dari jam unik, bukan dari baris.** Sembilan stasiun punya jam yang sama. Bila baris gabungan dipotong langsung, satu jam yang sama bisa jatuh di data latih (stasiun A) dan di data validasi (stasiun B), padahal PM2.5 antarstasiun berkorelasi tinggi pada jam yang sama (02.2). Dengan lipatan dari jam unik, semua stasiun mengikuti lipatan jam yang sama.

Sel di bawah membuat lipatan dan memeriksa aturan nomor 4, 5, dan 6.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Jam unik yang targetnya (t+24) masih di periode latih. Sama dengan jam baris yang dipakai model.
jam_cv = pd.DatetimeIndex(np.sort(latih["waktu"].unique()))
jam_cv = jam_cv[jam_cv < BATAS - pd.Timedelta(hours=HORIZON)]
cv = TimeSeriesSplit(n_splits=5, gap=24)          # jam_cv berurutan per jam, jadi gap=24 berarti 24 jam
lipatan = [(jam_cv[a], jam_cv[b]) for a, b in cv.split(jam_cv)]

for i, (a, b) in enumerate(lipatan, 1):
    assert a.max() + pd.Timedelta(hours=HORIZON) < b.min()       # target latih terakhir jatuh sebelum validasi dimulai
    assert len(a.intersection(b)) == 0                           # tidak ada jam yang sama di latih dan validasi
    print(f"Fold {i}: latih s.d. {a[-1]:%Y-%m-%d} | validasi {b[0]:%Y-%m-%d} s.d. {b[-1]:%Y-%m-%d} "
          f"| {len(a):,} jam latih, {len(b):,} jam validasi (x {len(STASIUN_LATIH)} stasiun)")
print("Data uji mulai", BATAS, "(tidak dipakai di fold mana pun)")

### 03.1 Mengapa skema ini
| Pilihan | Alasan |
|---|---|
| Uji = 20% jam terakhir, tanpa diacak | Data berurutan waktu. Bila diacak, model "mengintip" masa depan. Panduan menetapkan skema ini untuk struktur data seperti ini. |
| Satu batas waktu untuk semua stasiun | PM2.5 antarstasiun berkorelasi tinggi pada jam yang sama. Bila stasiun dipotong sendiri-sendiri, jam yang sama bisa ada di data latih satu stasiun dan di data uji stasiun lain. |
| 9 stasiun latih, 3 stasiun asing | Dua pertanyaan diuji terpisah: apakah model meramal baik di stasiun yang dikenal, dan apakah tetap baik di stasiun yang belum pernah dilihat (pergeseran distribusi). Periodenya sama, jadi selisih skor kedua kelompok berasal dari stasiun, bukan dari waktu. |
| `TimeSeriesSplit`, jendela latih melebar | Tiap *fold* melatih dengan semua masa lalu yang ada dan memvalidasi dengan periode sesudahnya, persis seperti pemakaian nyata (meramal ke depan). |
| 5 *fold* | Contoh di panduan. Tiap *fold* validasi sekitar enam setengah bulan, bergantian mencakup musim dingin dan musim hangat, jadi satu skor tidak hanya mewakili satu musim. |
| `gap=24` pada jam unik | Target sebuah baris adalah PM2.5 24 jam sesudahnya. Tanpa jarak, target baris latih terakhir jatuh di periode validasi. |
| Data uji dibuka satu kali | Setiap keputusan (fitur, model, pengaturan) dibuat dari validasi data latih. Skor uji hanya dilaporkan, tidak dipakai untuk menyetel ulang. |

Risiko kebocoran dan penjagaannya:

| Risiko | Penjagaan di *notebook* |
|---|---|
| Fitur memakai nilai masa depan | Fitur hanya sampai jam *t*, tanpa interpolasi, dihitung per stasiun (04.1). Diuji dengan memotong deret di tengah dan membandingkan fitur (04.7) |
| Target latih jatuh di periode uji | Baris latih 24 jam terakhir dibuang, dicek dengan `assert` (04.2) |
| Target latih jatuh di periode validasi | `gap=24` pada jam unik, dicek dengan `assert` (03 dan 04.5) |
| Jam yang sama ada di latih dan validasi lewat stasiun lain | Lipatan dari jam unik, dicek dengan `assert` (03) |
| Stasiun asing ikut terlatih | Hanya stasiun latih yang masuk `fitur_latih`, dicek dengan `assert` (04.2 dan 04.7) |
| Praproses belajar dari data validasi atau uji | Imputer dan *scaler* di dalam *pipeline*, dilatih ulang tiap *fold* (04.3, 04.5) |
| Pemilihan stasiun asing memakai data uji | Hanya data sebelum batas waktu yang dipakai (02.2) |
| Data uji dipakai berulang | Disisihkan di 02.2, dibuka satu kali di tahap 6 (evaluasi final) lewat penjaga di 03.3 |

### 03.2 Keterbatasan skema ini
* **Lima *fold* itu sedikit.** Simpangan baku RMSE antar *fold* besar (lihat 04.5), jadi selisih kecil antar model tidak boleh dianggap nyata.
* **Periode uji hanya sekitar 9,5 bulan** (2016-05-12 sampai 2017-02-28), tanpa Maret sampai awal Mei. Skor uji belum tentu mewakili satu siklus musim penuh.
* **Hanya satu pembagian uji.** Skor uji adalah satu hasil, bukan rata-rata dari beberapa periode.
* **Tiga stasiun asing hanya tiga contoh,** dipilih dari data periode latih (02.2). Selisih skor antara kelompok 9 dan kelompok 3 stasiun bisa dipengaruhi oleh stasiun mana yang terpilih, dan tidak boleh dibaca sebagai perkiraan untuk stasiun mana pun.
* **Stasiun Beijing saling berkorelasi tinggi** pada jam yang sama (angkanya dicetak di 02.2). Sembilan stasiun pada satu jam bukan sembilan sampel yang bebas, dan stasiun asing yang berdekatan dengan stasiun latih lebih mudah diramal daripada kota lain yang polanya tidak mirip.
* **Identitas stasiun tidak dipakai sebagai fitur** (keputusan LIP 16 Nov 2026). Stasiun asing tidak punya kategori yang dipelajari model, jadi fitur itu tidak bisa dipakai di sana. Model hanya melihat polutan, cuaca, dan kalender.
* **Beda skor antar *fold* belum diuji penyebabnya.** Musim hanya salah satu dugaan.

In [ ]:
# Pemeriksaan skema: tiga kelompok data tidak saling bocor
assert not set(STASIUN_LATIH) & set(STASIUN_ASING)
assert latih["station"].isin(STASIUN_LATIH).all() and latih["waktu"].max() < BATAS
print("Stasiun latih  :", ", ".join(STASIUN_LATIH))
print("Stasiun asing:", ", ".join(STASIUN_ASING))

### 03.3 Penjaga data uji
Berkas fitur uji hanya boleh dibaca lewat fungsi di bawah (dipakai nanti di tahap 6). Fungsi ini **tidak dijalankan di sini**.

* Pembukaan pertama dicatat di `Hasil/Tabel/log_pembukaan_data_uji.csv`: waktu, nama berkas, *hash* berkas, *hash* konfigurasi model, dan alasan.
* Membaca ulang dengan konfigurasi model yang **sama** diizinkan, supaya *notebook* tetap bisa dijalankan ulang dari awal.
* Membuka lagi dengan konfigurasi yang **berbeda** ditolak. Itu tanda model disetel ulang setelah melihat skor uji.

**Batas penjaga ini [jujur]:** hanya mencegah pembukaan ganda yang tidak disengaja. Orang yang menghapus baris log atau mengubah fungsi tetap bisa melewatinya, dan fungsi tidak bisa tahu apakah konfigurasi dipilih dari validasi atau dari skor uji. Aturan pemilihan model ada di `Laporan/Protokol Evaluasi Data Uji.md`, dan log ini baru berarti kalau di-*commit* bersama versi kode model yang dibekukan.

In [ ]:
import json, hashlib

LOG_UJI = os.path.join(TABEL, "log_pembukaan_data_uji.csv")
KOLOM_LOG = ["waktu", "berkas", "sha256_berkas", "sha256_konfigurasi", "alasan"]

def _sha(b):
    return hashlib.sha256(b).hexdigest()

def buka_data_uji(jalur, konfigurasi, alasan, log=LOG_UJI):
    """Satu-satunya pintu membaca berkas fitur uji. konfigurasi = dict hiperparameter, daftar fitur, dan praproses model final."""
    nama = os.path.basename(jalur)
    sha_konf = _sha(json.dumps(konfigurasi, sort_keys=True, default=str).encode())
    sha_berkas = _sha(open(jalur, "rb").read())
    riwayat = pd.read_csv(log) if os.path.exists(log) else pd.DataFrame(columns=KOLOM_LOG)
    lalu = riwayat[riwayat["berkas"] == nama]
    if len(lalu) == 0:
        baris = pd.DataFrame([[pd.Timestamp.now().isoformat(timespec="seconds"), nama,
                               sha_berkas, sha_konf, alasan]], columns=KOLOM_LOG)
        pd.concat([riwayat, baris], ignore_index=True).to_csv(log, index=False)
        print(f"Data uji {nama} dibuka untuk pertama kali dan dicatat.")
    elif sha_konf in set(lalu["sha256_konfigurasi"]):
        print(f"Data uji {nama} sudah pernah dibuka dengan konfigurasi yang sama: dibaca ulang untuk reproduksi.")
        if sha_berkas not in set(lalu["sha256_berkas"]):
            print("PERINGATAN: isi berkas berbeda dari saat pertama dibuka (cek versi pustaka atau apakah berkas dibuat ulang).")
    else:
        raise RuntimeError(f"{nama} sudah pernah dibuka dengan konfigurasi model yang lain. "
                           "Menyetel ulang model setelah melihat skor uji tidak diizinkan; laporkan skor pertama apa adanya.")
    return pd.read_csv(jalur, index_col=0, parse_dates=True)

print("Penjaga data uji siap. Belum ada berkas uji yang dibuka." if not os.path.exists(LOG_UJI)
      else "Log pembukaan data uji sudah ada:", LOG_UJI if os.path.exists(LOG_UJI) else "")

## 04. Fitur dan *pipeline* praproses
### 04.1 Siapkan fitur
Setiap baris = satu jam *t* di satu stasiun. **Target** = PM2.5 pada jam *t+24* di stasiun yang sama.
Semua fitur hanya memakai data sampai jam *t*, dan dihitung **per stasiun** (*lag* dan rata-rata bergulir tidak pernah mengambil nilai dari stasiun lain).

| Kelompok | Isi |
|---|---|
| Kondisi sekarang | polutan dan cuaca pada jam *t* |
| *Lag* | PM2.5 beberapa jam sebelumnya |
| *Rolling mean* | rata-rata PM2.5 6, 24, dan 72 jam terakhir |
| Kalender | jam, hari dalam pekan, bulan (dalam bentuk sin dan cos agar jam 23 dekat dengan jam 0) |
| Angin | arah angin diubah menjadi sin dan cos |

Nama stasiun disimpan sebagai kolom `stasiun` untuk keperluan pelaporan, **bukan sebagai fitur**.

**Nilai hilang [Usulan, bukan aturan dosen]:**
* Target hilang: baris dibuang, tidak diisi tebakan.
* Fitur hilang: **tidak diisi di sini** (tanpa *forward fill*). Semua yang kosong diurus imputer median di dalam *pipeline* (04.3), yang hanya belajar dari data latih tiap *fold*.

Mengapa tanpa *forward fill*? Keputusan 15 Nov 2026 dibuat dari perbandingan pada pilot Tiantan: pengisian 0, 1, 3, 6, 12, dan 24 jam tidak memberi beda yang bisa dibedakan dari derau antar *seed*. Di 04.6 perbandingan itu diulang pada 9 stasiun latih. Bila hasilnya menunjukkan beda yang jelas melebihi derau, keputusan itu perlu ditinjau.

In [ ]:
FFILL_MAKS = 0      # batas forward fill (jam). 0 = tanpa pengisian (keputusan 15 Nov 2026)
ARAH = {a: i * 22.5 for i, a in enumerate(
    ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"])}

def buat_fitur(data_stasiun):
    # Satu stasiun saja. Semua operasi memakai shift positif atau jendela ke belakang.
    d = data_stasiun.set_index("waktu").sort_index()
    f = pd.DataFrame(index=d.index)

    # Target: PM2.5 asli 24 jam ke depan (tanpa pengisian)
    f["target"] = d["PM2.5"].shift(-HORIZON)

    # Kondisi sekarang. Bila FFILL_MAKS > 0, celah pendek diisi nilai terakhir (hanya melihat ke belakang)
    isi = d[KOLOM_ANGKA].ffill(limit=FFILL_MAKS) if FFILL_MAKS > 0 else d[KOLOM_ANGKA]
    for k in KOLOM_ANGKA:
        f[k] = isi[k]

    # Lag dan rolling mean PM2.5 (shift positif = masa lalu)
    for j in [1, 2, 3, 6, 12, 24, 48]:
        f[f"PM2.5_lag{j}"] = isi["PM2.5"].shift(j)
    for w in [6, 24, 72]:
        f[f"PM2.5_rata{w}j"] = isi["PM2.5"].rolling(w, min_periods=w // 2).mean()

    # Kalender dan arah angin dalam bentuk lingkaran
    wd = d["wd"].ffill(limit=FFILL_MAKS) if FFILL_MAKS > 0 else d["wd"]
    sudut = wd.map(ARAH)
    f["angin_sin"], f["angin_cos"] = np.sin(np.radians(sudut)), np.cos(np.radians(sudut))
    f["jam_sin"], f["jam_cos"] = np.sin(2*np.pi*d.index.hour/24), np.cos(2*np.pi*d.index.hour/24)
    f["hari_sin"], f["hari_cos"] = np.sin(2*np.pi*d.index.dayofweek/7), np.cos(2*np.pi*d.index.dayofweek/7)
    f["bulan_sin"], f["bulan_cos"] = np.sin(2*np.pi*d.index.month/12), np.cos(2*np.pi*d.index.month/12)
    return f

def fitur_stasiun(data, daftar):
    # Fitur dihitung terpisah untuk setiap stasiun, lalu ditumpuk. Kolom `stasiun` bukan fitur.
    bagian = []
    for s in daftar:
        f = buat_fitur(data[data["station"] == s])
        f.insert(0, "stasiun", s)
        bagian.append(f)
    return pd.concat(bagian)

### 04.2 Potong fitur di batas data uji
Fitur dibuat dari deret lengkap tiap stasiun (karena hanya melihat ke belakang, ini aman), lalu dipotong di `BATAS` dan menurut stasiun:

* **Latih:** hanya 9 stasiun latih, hanya baris yang targetnya (jam *t+24*) jatuh sebelum `BATAS`.
* **Uji:** baris jam *t* di `BATAS` atau sesudahnya, dari 12 stasiun, diberi label kelompok (`stasiun_latih` atau `stasiun_asing`).
* **Dibuang:** baris 3 stasiun asing sebelum `BATAS` (tidak dipakai), dan baris 24 jam terakhir sebelum `BATAS` di semua stasiun.

**Celah kebocoran yang ditutup:** baris latih 24 jam terakhir punya target yang jatuh di periode uji.
Baris itu dibuang supaya tidak ada nilai uji yang ikut dilatih.

In [ ]:
fitur_semua = fitur_stasiun(df, STASIUN_LATIH + STASIUN_ASING)
waktu_target = fitur_semua.index + pd.Timedelta(hours=HORIZON)
ada_target = fitur_semua["target"].notna()

fitur_latih = fitur_semua[fitur_semua["stasiun"].isin(STASIUN_LATIH) & (waktu_target < BATAS) & ada_target]
fitur_uji = fitur_semua[(fitur_semua.index >= BATAS) & ada_target].copy()
fitur_uji.insert(1, "kelompok_uji", np.where(fitur_uji["stasiun"].isin(STASIUN_ASING), "stasiun_asing", "stasiun_latih"))
del fitur_semua

KOLOM_FITUR = [c for c in fitur_latih.columns if c not in ("stasiun", "target")]
n_asing = (fitur_uji["kelompok_uji"] == "stasiun_asing").sum()
print(f"Fitur latih: {len(fitur_latih):,} baris, {len(KOLOM_FITUR)} fitur, {fitur_latih['stasiun'].nunique()} stasiun")
print(f"Fitur uji  : {len(fitur_uji):,} baris (disimpan, tidak dibuka): "
      f"{len(fitur_uji) - n_asing:,} stasiun latih + {n_asing:,} stasiun asing")
print("Target latih terakhir:", (fitur_latih.index.max() + pd.Timedelta(hours=HORIZON)),
      "| harus sebelum", BATAS)

assert fitur_latih.index.max() + pd.Timedelta(hours=HORIZON) < BATAS
assert set(fitur_latih["stasiun"]) == set(STASIUN_LATIH)                 # tidak ada baris stasiun asing di data latih
assert fitur_uji.index.min() >= BATAS
assert set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_asing", "stasiun"]) == set(STASIUN_ASING)
assert set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_latih", "stasiun"]) == set(STASIUN_LATIH)

In [ ]:
# Cek: berapa persen fitur yang masih kosong (tanpa pengisian, diurus imputer di pipeline)
sisa = fitur_latih[KOLOM_FITUR].isna().mean().mul(100)
print(f"Fitur dengan nilai kosong tersisa: {(sisa > 0).sum()} dari {len(sisa)}, "
      f"tertinggi {sisa.idxmax()} = {sisa.max():.2f}%")

### 04.3 *Pipeline* praproses
Langkah yang "belajar" dari data (mengisi kosong dengan median, menyamakan skala) ditaruh di dalam *pipeline*.
*Pipeline* ini dilatih ulang di setiap *fold* `TimeSeriesSplit`, jadi hanya belajar dari data latih *fold* itu.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Belum dilatih di sini. Di tahap 4, pipeline ini dilatih ulang di setiap fold (cv dari bagian 03).
praproses = Pipeline([("isi_kosong", SimpleImputer(strategy="median")),
                      ("skala", StandardScaler())])
praproses

### 04.4 Simpan hasil
Dua berkas CSV: `fitur_latih.csv` (9 stasiun latih, periode latih) dan `fitur_uji.csv` (12 stasiun, periode uji, dengan kolom `kelompok_uji`). Berkas uji hanya disimpan.

In [ ]:
fitur_latih.rename_axis("waktu").to_csv(os.path.join(OLAHAN, "fitur_latih.csv"))
fitur_uji.rename_axis("waktu").to_csv(os.path.join(OLAHAN, "fitur_uji.csv"))
for nama_folder, isi in (("Gambar", GAMBAR), ("Tabel", TABEL), ("Fitur", OLAHAN)):
    print(f"{nama_folder}:", ", ".join(sorted(os.listdir(isi))))

### 04.5 *Baseline* (hasil *cross-validation*)
*Baseline* adalah titik acuan. Tanpanya, sebuah angka RMSE tidak bermakna.

| Model | Isi | Asal |
|---|---|---|
| *Naive forecast* | PM2.5 24 jam lagi dianggap sama dengan PM2.5 sekarang, di stasiun yang sama | Wajib untuk deret waktu (panduan) |
| Regresi linear | *Pipeline* 04.3 + regresi linear pada semua fitur, satu model untuk 9 stasiun latih | Pembanding tambahan [Usulan] |

Aturan evaluasi:
* `TimeSeriesSplit(n_splits=5, gap=24)` pada jam unik data latih (03). Data uji dan stasiun asing tidak disentuh.
* Skor tiap *fold* dihitung pada baris validasi dari 9 stasiun digabung.
* *Naive forecast* butuh PM2.5 jam *t*. Baris validasi yang PM2.5 jam *t*-nya kosong dibuang dari **kedua** model supaya dibandingkan pada baris yang sama.
* Hasil dilaporkan sebagai rata-rata ± simpangan baku antar *fold*. RMSE jadi metrik utama, MAE pendamping.

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LinearRegression

X = fitur_latih[KOLOM_FITUR]
y = fitur_latih["target"]

def rmse(y_asli, y_tebak): return float(np.sqrt(np.mean((y_asli - y_tebak) ** 2)))
def mae(y_asli, y_tebak):  return float(np.mean(np.abs(y_asli - y_tebak)))

nama_model = ["Naive forecast", "Regresi linear"]
skor = {n: {"RMSE": [], "MAE": []} for n in nama_model}
dibuang = jumlah_validasi = 0
for jam_a, jam_b in lipatan:
    # target baris latih harus jatuh sebelum periode validasi dimulai, dan tidak ada jam yang dipakai bersama
    assert jam_a.max() + pd.Timedelta(hours=HORIZON) < jam_b.min()
    a, b = X.index.isin(jam_a), X.index.isin(jam_b)
    ada = X[b]["PM2.5"].notna().to_numpy()           # naive butuh PM2.5 jam t
    dibuang += int((~ada).sum()); jumlah_validasi += int(b.sum())
    Xv, yv = X[b][ada], y[b][ada].to_numpy()

    tebak = {"Naive forecast": Xv["PM2.5"].to_numpy()}
    model = Pipeline([("praproses", clone(praproses)), ("model", LinearRegression())])
    tebak["Regresi linear"] = model.fit(X[a], y[a]).predict(Xv)

    for n in nama_model:
        skor[n]["RMSE"].append(rmse(yv, tebak[n]))
        skor[n]["MAE"].append(mae(yv, tebak[n]))

tabel_baseline = pd.DataFrame({
    "RMSE rata-rata": [np.mean(skor[n]["RMSE"]) for n in nama_model],
    "RMSE sd":        [np.std(skor[n]["RMSE"], ddof=1) for n in nama_model],
    "MAE rata-rata":  [np.mean(skor[n]["MAE"]) for n in nama_model],
    "MAE sd":         [np.std(skor[n]["MAE"], ddof=1) for n in nama_model]}, index=nama_model).round(2)
tabel_baseline.to_csv(os.path.join(TABEL, "t4_tabel_baseline.csv"))
print(f"Baris validasi dibuang (PM2.5 jam t kosong): {dibuang} dari {jumlah_validasi:,}")
print("RMSE per fold:")
for n in nama_model:
    print(f"  {n:<15}", np.round(skor[n]["RMSE"], 1))
tabel_baseline

### 04.6 Dasar keputusan: batas *forward fill*
*Bagian pendukung: boleh dilewati saat membaca pertama.* Isinya hanya pembenaran satu keputusan (tanpa *forward fill*), tidak dipakai bagian lain.

Keputusan 15 Nov 2026: tanpa *forward fill*, diambil dari pilot Tiantan. Di sini dasarnya dihitung ulang pada **9 stasiun latih**, **hanya dari data latih** (data uji dan stasiun asing tidak dipakai). Fitur dibuat ulang per stasiun dengan batas pengisian 0, 1, 3, 6, 12, dan 24 jam, lalu dua model dinilai dengan lipatan jam unik yang sama dengan bagian 03.
Karena celah pendek diisi nilai terakhir (hanya melihat ke belakang), tidak ada nilai masa depan yang masuk.

Yang dibandingkan adalah selisih antar pengaturan *dan* derau antar *seed* pada pengaturan yang sama. Selisih yang lebih kecil dari derau tidak boleh dianggap nyata.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

def fitur_dengan_ffill(ff):
    # Dihitung dari `latih` (9 stasiun latih, sebelum BATAS) saja. Target 24 jam terakhir otomatis kosong, jadi dibuang.
    global FFILL_MAKS
    simpan, FFILL_MAKS = FFILL_MAKS, ff
    try:
        return fitur_stasiun(latih, STASIUN_LATIH)
    finally:
        FFILL_MAKS = simpan

def model_uji(nama, seed=SEED):
    reg = Ridge(alpha=1.0) if nama == "Ridge" else HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.1, max_depth=4, random_state=seed)
    return Pipeline([("isi", SimpleImputer(strategy="median")), ("skala", StandardScaler()), ("m", reg)])

def baris_fold(F, jam):
    x = F[F.index.isin(jam)].dropna(subset=["target"])
    return x[KOLOM_FITUR], x["target"]

daftar_ff = [0, 1, 3, 6, 12, 24]
skor_ff = {}
for ff in daftar_ff:
    F = fitur_dengan_ffill(ff)
    for nm in ["Ridge", "HGB"]:
        r = []
        for a, b in lipatan:
            Xa, ya = baris_fold(F, a); Xb, yb = baris_fold(F, b)
            r.append(rmse(yb, model_uji(nm).fit(Xa, ya).predict(Xb)))
        skor_ff[(ff, nm)] = np.array(r)

baris_tabel = []
for nm in ["Ridge", "HGB"]:
    dasar = skor_ff[(3, nm)]
    for ff in daftar_ff:
        r = skor_ff[(ff, nm)]
        baris_tabel.append({"model": nm, "ffill_maks_jam": ff, "RMSE rata-rata": r.mean(), "RMSE sd antar fold": r.std(ddof=1),
                            "selisih vs 3 jam": (r - dasar).mean(), "fold lebih baik dari 3 jam": int(((r - dasar) < 0).sum())})
tabel_ff = pd.DataFrame(baris_tabel).round(3)
tabel_ff.to_csv(os.path.join(TABEL, "t2_tabel_ffill.csv"), index=False)
tabel_ff

In [ ]:
# Lantai derau: pengaturan yang sama (ffill = 3 jam), hanya seed HGB yang diganti
F3 = fitur_dengan_ffill(3)
rata_seed = []
for sd in range(6):
    r = []
    for a, b in lipatan:
        Xa, ya = baris_fold(F3, a); Xb, yb = baris_fold(F3, b)
        r.append(rmse(yb, model_uji("HGB", seed=sd).fit(Xa, ya).predict(Xb)))
    rata_seed.append(np.mean(r))
rentang_seed = max(rata_seed) - min(rata_seed)
rentang_ff = {nm: float(np.ptp([skor_ff[(ff, nm)].mean() for ff in daftar_ff])) for nm in ["Ridge", "HGB"]}
sd_fold = {nm: float(np.mean([skor_ff[(ff, nm)].std(ddof=1) for ff in daftar_ff])) for nm in ["Ridge", "HGB"]}
print("RMSE rata-rata per seed HGB (ffill = 3 jam):", np.round(rata_seed, 2), "| rentang", round(rentang_seed, 2))
print("Rentang rata-rata RMSE antar pengaturan ffill :", {k: round(v, 2) for k, v in rentang_ff.items()})
print("Simpangan baku rata-rata antar fold           :", {k: round(v, 2) for k, v in sd_fold.items()})
print("Baca: rentang antar pengaturan yang tidak melebihi simpangan baku antar fold (dan, untuk HGB, rentang seed) "
      "tidak boleh dianggap beda nyata.")

### 04.7 Pemeriksaan kesesuaian dengan desain 9/3
Sel di bawah memeriksa desain pada data yang benar-benar dibuat di atas. Semua pemeriksaan memakai `assert`, jadi *notebook* berhenti bila ada yang dilanggar.

1. Stasiun latih dan stasiun asing tidak beririsan, jumlahnya 9 dan 3.
2. Berkas latih hanya berisi stasiun latih, dan target barisnya jatuh sebelum `BATAS`.
3. Berkas uji berisi jam sejak `BATAS`, untuk dua kelompok, dan stasiun asing **hanya** ada di berkas uji.
4. Lipatan validasi: jarak aman minimal 24 jam dan tidak ada jam yang sama di latih dan validasi.
5. **Fitur tidak melihat masa depan:** fitur satu stasiun dihitung dari deret yang dipotong di tengah, lalu dibandingkan dengan fitur dari deret penuh pada jam yang sama. Keduanya harus sama persis (target dikecualikan, karena target memang melihat ke depan).
6. Status penjaga data uji.

In [ ]:
# 1
assert len(STASIUN_LATIH) == 9 and len(STASIUN_ASING) == 3 and not set(STASIUN_LATIH) & set(STASIUN_ASING)
# 2
assert set(fitur_latih["stasiun"]) == set(STASIUN_LATIH)
assert (fitur_latih.index + pd.Timedelta(hours=HORIZON) < BATAS).all() and fitur_latih["target"].notna().all()
# 3
assert (fitur_uji.index >= BATAS).all() and fitur_uji["target"].notna().all()
assert set(fitur_uji["kelompok_uji"]) == {"stasiun_latih", "stasiun_asing"}
assert not set(fitur_latih["stasiun"]) & set(fitur_uji.loc[fitur_uji["kelompok_uji"] == "stasiun_asing", "stasiun"])
# 4
for a, b in lipatan:
    assert a.max() + pd.Timedelta(hours=HORIZON) < b.min() and len(a.intersection(b)) == 0
# 5
T0 = pd.Timestamp("2015-06-01 00:00")
for s in (STASIUN_LATIH[0], STASIUN_ASING[0]):
    penuh = buat_fitur(df[df["station"] == s]).loc[:T0, KOLOM_FITUR]
    potong = buat_fitur(df[(df["station"] == s) & (df["waktu"] <= T0)]).loc[:T0, KOLOM_FITUR]
    pd.testing.assert_frame_equal(penuh, potong)
    print(f"Fitur {s} sampai {T0} identik dengan atau tanpa data sesudahnya ({len(penuh):,} baris diperiksa).")
# 6
print("Penjaga data uji:", "BELUM ADA berkas uji yang dibuka." if not os.path.exists(LOG_UJI)
      else f"log pembukaan ada di {LOG_UJI}")
print("Semua pemeriksaan desain 9/3 lolos.")

## Ringkasan
Angka di bawah dihitung langsung dari data latih di atas.

In [ ]:
n_uji = fitur_uji["kelompok_uji"].value_counts()
print(f"1. Data latih sampai {BATAS - pd.Timedelta(hours=1)}, data uji mulai {BATAS} (tidak dibuka), batas sama untuk 12 stasiun.")
print(f"2. Stasiun latih ({len(STASIUN_LATIH)}): {', '.join(STASIUN_LATIH)}.")
print(f"   Stasiun asing ({len(STASIUN_ASING)}, dikunci LIP, sama dengan aturan urutan jarak distribusi {URUTAN_ASING}): {', '.join(STASIUN_ASING)}.")
print(f"3. PM2.5 data latih (9 stasiun): median {pm.median():.0f}, rata-rata {pm.mean():.0f}, miring ke kanan "
      f"(skewness {pm.skew():.2f}). Galat besar saat episode tinggi akan mendominasi RMSE.")
print(f"4. Nilai hilang PM2.5 {latih['PM2.5'].isna().mean():.1%}; {(celah <= 3).mean():.0%} celah "
      f"<= 3 jam, terpanjang {celah.max()} jam. Tidak diisi (FFILL_MAKS = {FFILL_MAKS}), diurus imputer di pipeline.")
print(f"5. Pola: tertinggi bulan {nama_bulan[per_bulan.idxmax()-1]}, jam {per_jam.idxmax()}:00.")
print(f"6. Korelasi PM2.5 jam t dengan jam t+24: {korelasi['PM2.5']:.2f}. "
      f"Fitur terkuat lain: {korelasi.drop('PM2.5').abs().idxmax()} ({korelasi.drop('PM2.5')[korelasi.drop('PM2.5').abs().idxmax()]:.2f}).")
print(f"7. {len(KOLOM_FITUR)} fitur siap. Baris latih {len(fitur_latih):,}; baris uji {len(fitur_uji):,} "
      f"({n_uji['stasiun_latih']:,} stasiun latih, {n_uji['stasiun_asing']:,} stasiun asing).")
print(f"8. Baseline (CV 5 fold, 9 stasiun, RMSE rata-rata ± sd): "
      f"naive {tabel_baseline.loc['Naive forecast', 'RMSE rata-rata']:.1f} ± {tabel_baseline.loc['Naive forecast', 'RMSE sd']:.1f}, "
      f"regresi linear {tabel_baseline.loc['Regresi linear', 'RMSE rata-rata']:.1f} ± {tabel_baseline.loc['Regresi linear', 'RMSE sd']:.1f}.")

**Yang perlu diputuskan atau dicek berikutnya**
* Tiga stasiun asing (Dingling, Wanliu, Shunyi) sudah dikunci LIP sebelum data uji dibuka. Mengubahnya berarti mengubah keputusan, dan *notebook* dijalankan ulang dari awal.
* Desain 9 stasiun latih dan 3 stasiun asing belum dikonfirmasi dosen.
* Tahap berikutnya: model lanjutan dan *tuning* (bagian 05), minimal dua model, dibandingkan dengan *baseline* di 04.5, memakai lipatan yang sama.
* Nilai ekstrem dipertahankan (keputusan 14 Nov 2026). Hasil 02.5 dipakai untuk mengecek apakah keputusan itu masih wajar.
* Berkas uji (`fitur_uji.csv`) dibuka satu kali di tahap 6 lewat `buka_data_uji`, dan skor dilaporkan terpisah untuk 9 stasiun latih dan 3 stasiun asing.